# Reranker Training Wrapper

Run this notebook on Google Colab or Kaggle.

This notebook only orchestrates `main/train_reranker.py`.

Workflow:
1. Install dependencies
2. Locate the repository root
3. Launch training
4. Package the best checkpoint

In [1]:
import sys
import torch

print("1. Python Interpreter hiện tại:")
print(sys.executable)
print("\n2. Phiên bản PyTorch của Notebook:")
print(torch.__version__)
print("\n3. CUDA (GPU) có khả dụng không?")
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print("Tên GPU nhận diện được:", torch.cuda.get_device_name(0))


1. Python Interpreter hiện tại:
d:\Data\File for Google Drive real\Project\Nutrition_RAG\HEALTHCARE-RAG\.venv\Scripts\python.exe

2. Phiên bản PyTorch của Notebook:
2.6.0+cu124

3. CUDA (GPU) có khả dụng không?
True
Tên GPU nhận diện được: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
import sys
import subprocess

packages = [
    "sentence-transformers>=4.0.0",
    "datasets",
    "rank-bm25",
    "scikit-learn",
    "accelerate",
    "tqdm",
    "pyyaml",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Dependencies ready.")

Dependencies ready.


In [3]:
# Lấy đường dẫn repo root khi chạy cục bộ (local offline)
from pathlib import Path
import sys

repo_root = Path.cwd().parent.parent.resolve()
sys.path.insert(0, str(repo_root))
print("Local Repo root:", repo_root)

Local Repo root: D:\Data\File for Google Drive real\Project\Nutrition_RAG\HEALTHCARE-RAG


In [3]:
from pathlib import Path
import sys

def in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    candidate_roots = [
        Path("/content/drive/MyDrive/nutrition-rag/HEALTHCARE-RAG"),
        Path("/content/drive/MyDrive/HEALTHCARE-RAG"),
        Path("/content/HEALTHCARE-RAG"),
    ]
elif Path("/kaggle/working/HEALTHCARE-RAG").exists():
    candidate_roots = [
        Path("/kaggle/working/HEALTHCARE-RAG"),
        Path("/kaggle/input/HEALTHCARE-RAG"),
    ]
else:
    candidate_roots = [Path.cwd(), Path.cwd() / "HEALTHCARE-RAG"]

repo_root = next((
    path.resolve()
    for path in candidate_roots
    if (path / "main" / "train_reranker.py").exists()
), None)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repo root. Move the repository to one of the candidate paths or set REPO_ROOT manually."
    )

sys.path.insert(0, str(repo_root))
print("Repo root:", repo_root)

FileNotFoundError: Could not find the repo root. Move the repository to one of the candidate paths or set REPO_ROOT manually.

In [4]:
REPO_ROOT = repo_root
TRAIN_SCRIPT = REPO_ROOT / "main" / "train_reranker.py"
OUTPUT_DIR = REPO_ROOT / "models" / "reranker_domain"

BASE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 3e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
REPORT_TO = "none"
TRAIN_HARD_NEGATIVES_PER_QUERY = 6
TRAIN_RANDOM_NEGATIVES_PER_QUERY = 2

TRAIN_QRELS = REPO_ROOT / "data/nfcorpus/qrels/train.tsv"
DEV_QRELS = REPO_ROOT / "data/nfcorpus/qrels/dev.tsv"
QUERIES_PATH = REPO_ROOT / "data/nfcorpus/queries.jsonl"
CORPUS_PATH = REPO_ROOT / "data/en/corpus.jsonl"

for path in [TRAIN_SCRIPT, TRAIN_QRELS, DEV_QRELS, QUERIES_PATH, CORPUS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Training script:", TRAIN_SCRIPT)
print("Output dir:", OUTPUT_DIR)
print("Train qrels:", TRAIN_QRELS)
print("Dev qrels:", DEV_QRELS)

Training script: D:\Data\File for Google Drive real\Project\Nutrition_RAG\HEALTHCARE-RAG\main\train_reranker.py
Output dir: D:\Data\File for Google Drive real\Project\Nutrition_RAG\HEALTHCARE-RAG\models\reranker_domain
Train qrels: D:\Data\File for Google Drive real\Project\Nutrition_RAG\HEALTHCARE-RAG\data\nfcorpus\qrels\train.tsv
Dev qrels: D:\Data\File for Google Drive real\Project\Nutrition_RAG\HEALTHCARE-RAG\data\nfcorpus\qrels\dev.tsv


In [5]:
import subprocess
import sys

command = [
    sys.executable,
    str(TRAIN_SCRIPT),
    "--repo-root", str(REPO_ROOT),
    "--output-dir", str(OUTPUT_DIR),
    "--base-model", BASE_MODEL,
    "--epochs", str(EPOCHS),
    "--train-batch-size", str(TRAIN_BATCH_SIZE),
    "--eval-batch-size", str(EVAL_BATCH_SIZE),
    "--learning-rate", str(LEARNING_RATE),
    "--warmup-ratio", str(WARMUP_RATIO),
    "--weight-decay", str(WEIGHT_DECAY),
    "--report-to", REPORT_TO,
    "--train-hard-negatives-per-query", str(TRAIN_HARD_NEGATIVES_PER_QUERY),
    "--train-random-negatives-per-query", str(TRAIN_RANDOM_NEGATIVES_PER_QUERY),
]

print("Training command:", command)
subprocess.run(command, check=True)

Training command: ['d:\\Data\\File for Google Drive real\\Project\\Nutrition_RAG\\HEALTHCARE-RAG\\.venv\\Scripts\\python.exe', 'D:\\Data\\File for Google Drive real\\Project\\Nutrition_RAG\\HEALTHCARE-RAG\\main\\train_reranker.py', '--repo-root', 'D:\\Data\\File for Google Drive real\\Project\\Nutrition_RAG\\HEALTHCARE-RAG', '--output-dir', 'D:\\Data\\File for Google Drive real\\Project\\Nutrition_RAG\\HEALTHCARE-RAG\\models\\reranker_domain', '--base-model', 'cross-encoder/ms-marco-MiniLM-L-6-v2', '--epochs', '3', '--train-batch-size', '16', '--eval-batch-size', '32', '--learning-rate', '3e-05', '--warmup-ratio', '0.1', '--weight-decay', '0.01', '--report-to', 'none', '--train-hard-negatives-per-query', '6', '--train-random-negatives-per-query', '2']


CompletedProcess(args=['d:\\Data\\File for Google Drive real\\Project\\Nutrition_RAG\\HEALTHCARE-RAG\\.venv\\Scripts\\python.exe', 'D:\\Data\\File for Google Drive real\\Project\\Nutrition_RAG\\HEALTHCARE-RAG\\main\\train_reranker.py', '--repo-root', 'D:\\Data\\File for Google Drive real\\Project\\Nutrition_RAG\\HEALTHCARE-RAG', '--output-dir', 'D:\\Data\\File for Google Drive real\\Project\\Nutrition_RAG\\HEALTHCARE-RAG\\models\\reranker_domain', '--base-model', 'cross-encoder/ms-marco-MiniLM-L-6-v2', '--epochs', '3', '--train-batch-size', '16', '--eval-batch-size', '32', '--learning-rate', '3e-05', '--warmup-ratio', '0.1', '--weight-decay', '0.01', '--report-to', 'none', '--train-hard-negatives-per-query', '6', '--train-random-negatives-per-query', '2'], returncode=0)

In [ ]:
import shutil

final_dir = OUTPUT_DIR / "final"
if not final_dir.exists():
    raise FileNotFoundError(f"Expected final checkpoint not found: {final_dir}")

archive_path = shutil.make_archive(str(final_dir), "zip", final_dir)
print("Final checkpoint:", final_dir)
print("Archive:", archive_path)

if in_colab():
    from google.colab import files
    files.download(archive_path)
else:
    print("Download the archive manually from the notebook file browser.")

# After training

The fine-tuned reranker lives in `models/reranker_domain/final/`.

To use it in the pipeline, set `reranker_model` in `configs/config.yaml` to that folder.